In [1]:
import pathlib
import types
from collections import namedtuple

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, auc
from sklearn.covariance import EmpiricalCovariance
import torch
import pandas as pd
from plotly import express as px
from jaxtyping import Array, Float, Int

import trapallada.model.ar

# Parameters

File with annotations

In [2]:
annotated_windows_file = pathlib.Path.home() / 'trapallada' / 'annotations' / 'annotated_windows.parquet'
assert annotated_windows_file.exists()

Directory with the pre-processed data (normalized windows)

In [3]:
data_dir = pathlib.Path.home() / 'trapallada' / 'data' / 'rates_large_infs'
assert data_dir.exists()
data_dir

PosixPath('/home/manu/trapallada/data/rates_large_infs')

Different files of interest

In [4]:
masks_file = data_dir / 'masks.pt'
metadata_file = data_dir / 'metadata.parquet'
signals_file = data_dir / 'signals.pt'

# Processing

Data is loaded...

In [5]:
signals: Float[Array, 'signal channel length'] = torch.from_numpy(torch.load(signals_file, weights_only=False))
masks: Float[Array, 'signal length'] = torch.load(masks_file, weights_only=False)
metadata_df = pd.read_parquet(metadata_file)
signals.shape, masks.shape, metadata_df.shape

(torch.Size([30856, 5, 225]), (30856, 225), (30856, 4))

...and packed into a *dummy* object...

In [6]:
packed_data = types.SimpleNamespace(signals=signals, masks=masks, metadata_df=metadata_df)

...that for the purposes of `ols_dataset_fit` acts as a `SignalsDataset` (what the former expects).

In [7]:
AR_coeffs = trapallada.model.ar.ols_dataset_fit(packed_data, p=5)
AR_coeffs.shape

(30856, 125)

Annotations are read and *tidie-up* a little

In [8]:
annotated_windows_df = pd.read_parquet(annotated_windows_file)
annotated_windows_df= annotated_windows_df[~pd.isna(annotated_windows_df['Label'])]
annotated_windows_df['Event'] = annotated_windows_df.eval('Label == "Interesting" & Type != "background"')
annotated_windows_df

,start,end,tar file,source,signal,Selected by,Label,Type,Subtype,Event
14796,2004-02-29 23:34:36.224518,2004-03-01 05:17:59.744518,0201360201.tar,P0201360201PNX000SRCTSR8002.FTZ,0201360201/P0201360201PNX000SRCTSR8002,VAE,Interesting,doubleExponential,flare,True
15937,2011-01-03 19:00:05.711776,2011-01-04 00:43:29.231776,0652330101.tar,P0652330101PNX000SRCTSR8001.FTZ,0652330101/P0652330101PNX000SRCTSR8001,VAE,Interesting,doubleExponential,flare,True
22425,2015-03-26 05:54:06.667239,2015-03-26 11:37:30.187239,0764800301.tar,P0764800301PNX000SRCTSR8005.FTZ,0764800301/P0764800301PNX000SRCTSR8005,NaN,Interesting,background,None,False
29039,2015-06-27 04:31:17.762896,2015-06-27 10:14:41.282896,0761970101.tar,P0761970101PNX000SRCTSR8002.FTZ,0761970101/P0761970101PNX000SRCTSR8002,NaN,NOT interesting,None,None,False
24571,2013-05-19 04:37:04.302948,2013-05-19 10:20:27.822948,0723700101.tar,P0723700101PNX000SRCTSR8002.FTZ,0723700101/P0723700101PNX000SRCTSR8002,AR,NOT interesting,None,None,False
...,...,...,...,...,...,...,...,...,...,...
27106,2014-10-04 23:21:51.077119,2014-10-05 05:05:14.597119,0728990101.tar,P0728990101PNX000SRCTSR8001.FTZ,0728990101/P0728990101PNX000SRCTSR8001,AR,NOT interesting,None,None,False
9253,2008-06-27 08:47:46.841629,2008-06-27 14:31:10.361629,0551750301.tar,P0551750301PNX000SRCTSR8001.FTZ,0551750301/P0551750301PNX000SRCTSR8001,AR,NOT interesting,None,None,False
15049,2017-10-24 15:17:34.836966,2017-10-24 21:00:58.356966,0795730101.tar,P0795730101PNX000SRCTSR8001.FTZ,0795730101/P0795730101PNX000SRCTSR8001,VAE,Interesting,background,None,False
5371,2013-09-27 21:06:30.736658,2013-09-28 02:49:54.256658,0724380101.tar,P0724380101PNX000SRCTSR8003.FTZ,0724380101/P0724380101PNX000SRCTSR8003,NaN,NOT interesting,None,None,False


The `Label` yields the desired class

In [9]:
y = annotated_windows_df['Event'].astype(int)
y

14796    1
15937    1
22425    0
29039    0
24571    0
        ..
27106    0
9253     0
15049    0
5371     0
24033    0
Name: Event, Length: 194, dtype: int64

A convenience function to evaluate the algorithms (as binary classifiers)

In [10]:
# so that named values are returned
MahalanobisBasedMetrics = namedtuple('MahalanobisBasedMetrics', ['average_precision', 'best_recall_at_min_precision', 'roc_curve'])

def mahalanobis_based_metrics(
        x: Float[Array, 'window feature'],
        labels: Int[Array, 'window'],
        i_windows_of_interest: Int[Array, 'window'],
        min_precision: float = 0.7,
        algorithm_name: str | None = None,
    ) -> MahalanobisBasedMetrics:

    # covariance matrix (and mean) is estimated...
    cov = EmpiricalCovariance().fit(x)

    # Mahalanobis distance (given the above covariance) from every sample to the mean (a measure of *outlierness*)
    distances = cov.mahalanobis(x)

    # the distances for the windows of interest are kept as scores
    scores = distances[i_windows_of_interest]

    roc_auc: float = roc_auc_score(labels, scores)
    average_precision: float = average_precision_score(labels, scores)

    precision, recall, thr = precision_recall_curve(labels, scores)
    mask = (precision >= min_precision)
    best_recall_at_min_precision: float = recall[mask].max().item()

    fpr, tpr, thresholds = roc_curve(labels, scores)

    fig = px.area(
        x=fpr, y=tpr,
        title=f'<b>{algorithm_name}</b> (AUC={auc(fpr, tpr):.2f}, AP={average_precision:.2f}, BR@{min_precision}={best_recall_at_min_precision:.2f})',
        labels=dict(x='False Positive Rate', y='True Positive Rate'),
        width=500, height=500,
    )
    fig.add_shape(
        type='line', line=dict(dash='dash'),
        x0=0, x1=1, y0=0, y1=1
    )

    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])

    return MahalanobisBasedMetrics(average_precision, best_recall_at_min_precision, fig)

In [11]:
res_ar = mahalanobis_based_metrics(AR_coeffs, y, annotated_windows_df.index.values, algorithm_name='AR')
res_ar[-1]